#1 - Importation des librairies et Chargement des données

##1.1 - Importation des librairies


In [155]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import plotly.graph_objects as go

##1.2 Chargement des données

In [156]:
df_transactions = pd.read_csv("Transactions.csv",sep=';')
df_customers = pd.read_csv("customers.csv",sep=';')
df_products = pd.read_csv("products.csv",sep=';')

/tmp/ipython-input-3238197900.py:1: DtypeWarning:

Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.



#2 - Analyse exploratoire des fichiers

##2.1 - Exploration rapide

In [157]:
df_transactions.info()

##Mise en évidence de lignes vides

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id_prod     687534 non-null  object
 1   date        687534 non-null  object
 2   session_id  687534 non-null  object
 3   client_id   687534 non-null  object
dtypes: object(4)
memory usage: 32.0+ MB


In [158]:
df_transactions.head()

##Format de date peu exploitable

,id_prod,date,session_id,client_id
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033


In [159]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8621 entries, 0 to 8620
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   client_id  8621 non-null   object
 1   sex        8621 non-null   object
 2   birth      8621 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 202.2+ KB


In [160]:
df_customers.head()

,client_id,sex,birth
0,c_4410,f,1967
1,c_7839,f,1975
2,c_1699,f,1984
3,c_5961,f,1962
4,c_5320,m,1943


In [161]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3286 entries, 0 to 3285
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id_prod  3286 non-null   object 
 1   price    3286 non-null   float64
 2   categ    3286 non-null   int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 77.1+ KB


In [162]:
df_products.head()

,id_prod,price,categ
0,0_1421,19.99,0
1,0_1368,5.13,0
2,0_731,17.99,0
3,1_587,4.99,1
4,0_1507,3.99,0


##2.2 - Corrections

In [163]:
#Suppression des lignes vides dans le fichier Transactions
df_transactions=df_transactions.dropna()

In [164]:
# Conversion de la colonne 'date' en datetime
df_transactions['date'] = pd.to_datetime(df_transactions['date'])

# Précision au mois (début du mois) en convertissant en période mensuelle puis en timestamp
df_transactions['date_mois'] = df_transactions['date'].dt.to_period('M').dt.to_timestamp()

# Précision à l'année (début d'année') en convertissant en période annuelle puis en timestamp
df_transactions['date_annee'] = df_transactions['date'].dt.to_period('Y').dt.to_timestamp()

# Précision au jour en convertissant en période journalière puis en timestamp
df_transactions['date_jour'] = df_transactions['date'].dt.to_period('d').dt.to_timestamp()

# Vérification du nouveau format
display(df_transactions.head())

,id_prod,date,session_id,client_id,date_mois,date_annee,date_jour
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,2021-03-01,2021-01-01,2021-03-01
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,2021-03-01,2021-01-01,2021-03-01
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,2021-03-01,2021-01-01,2021-03-01
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,2021-03-01,2021-01-01,2021-03-01
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,2021-03-01,2021-01-01,2021-03-01


In [165]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 687534 entries, 0 to 687533
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     687534 non-null  object        
 1   date        687534 non-null  datetime64[ns]
 2   session_id  687534 non-null  object        
 3   client_id   687534 non-null  object        
 4   date_mois   687534 non-null  datetime64[ns]
 5   date_annee  687534 non-null  datetime64[ns]
 6   date_jour   687534 non-null  datetime64[ns]
dtypes: datetime64[ns](4), object(3)
memory usage: 42.0+ MB


In [166]:
#Vérification des doublons
df_products.duplicated().any() or df_transactions.duplicated().any() or df_customers.duplicated().any()

np.False_

#3 - Analyses des KPI

##3.0 - Préparation des données

In [167]:
#Nouveau df joignant les transactions et les produits
df_transactions_produits=pd.merge(df_transactions, df_products, on='id_prod', how='left')

In [168]:
df_transactions_produits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687534 entries, 0 to 687533
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     687534 non-null  object        
 1   date        687534 non-null  datetime64[ns]
 2   session_id  687534 non-null  object        
 3   client_id   687534 non-null  object        
 4   date_mois   687534 non-null  datetime64[ns]
 5   date_annee  687534 non-null  datetime64[ns]
 6   date_jour   687534 non-null  datetime64[ns]
 7   price       687534 non-null  float64       
 8   categ       687534 non-null  int64         
dtypes: datetime64[ns](4), float64(1), int64(1), object(3)
memory usage: 47.2+ MB


##3.1 - CA et moyenne mobile

In [169]:
#Produit le moins cher
print("Prix minimum: {}".format(df_products["price"].min()))

#Produit le plus cher
print("Prix maximum: {}".format(df_products["price"].max()))

Prix minimum: 0.62
Prix maximum: 300.0


In [170]:
##Calcul du CA total
CA_total=df_transactions_produits["price"].sum()
print("Le CA total est de {} euros".format(round(CA_total)))

Le CA total est de 12027663 euros


In [171]:
##Groupby avec le CA mensuel
df_CA_mois=df_transactions_produits.groupby("date_mois")['price'].sum().reset_index()
df_CA_mois.rename(columns={'price' : 'CA'},inplace=True)

##Groupby avec le CA journalier
df_CA_jour=df_transactions_produits.groupby("date_jour")['price'].sum().reset_index()
df_CA_jour.rename(columns={'price' : 'CA'},inplace=True)

In [172]:
#Calcul de la moyenne mobile (sur un mois)
df_CA_jour['moyenne_mobile'] = df_CA_jour['CA'].rolling(window=30,center=True).mean()

In [173]:
#Barchart CA
fig311 = px.line(df_CA_jour, x="date_jour", y="CA", title='Evolution du CA dans le temps')
#Ajout de la moyenne mobile
fig311.add_trace(go.Scatter(x=df_CA_jour['date_jour'], y=df_CA_jour['moyenne_mobile'], mode='lines', name='Moyenne Mobile'))
fig311.update_layout(
    xaxis_title="Temps",
    yaxis_title="CA(€)"
)
fig311.show()

In [174]:
#Groupby dates avec aggrégé par somme des prix de vente
df_CA_annee=df_transactions_produits.groupby('date_annee')['price'].sum().reset_index()
df_CA_annee.rename(columns={'price' : 'CA'},inplace=True)

fig312 = px.bar(df_CA_annee, x="date_annee", y="CA", title='Evolution du CA dans le temps (annee)')
fig312.update_layout(
    xaxis_title="Temps",
    yaxis_title="CA(€)")
fig312.show()

##3.2 - CA par catégorie

In [175]:
#Groupby par date et par catégorie agrégé par CA
df_CA_categ=df_transactions_produits.groupby(['date_mois','categ'])['price'].sum().reset_index()

df_CA_categ.rename(columns={'price' : 'CA'},inplace=True)

In [176]:
#fig321 = px.histogram(df_CA_categ, x='date_mois', y='CA', color='categ', barmode='group', title='Evolution du CA par catégorie')
#fig321.update_layout(
#    xaxis_title="Temps",
#    yaxis_title="CA(€)",
#)
#fig321.show()

In [177]:
fig322 = px.line(df_CA_categ, x="date_mois", y="CA", color='categ', title='Evolution du CA par catégorie dans le temps')
fig322.show()

##3.3 - Nombre de clients uniques par mois

In [178]:
# Groupby par mois et nombre de clients uniques
df_nb_customers = df_transactions_produits.groupby('date_mois')['client_id'].nunique().reset_index()
df_nb_customers.rename(columns={'client_id' : 'Nombre de clients uniques'},inplace=True)
display(df_nb_customers)

,date_mois,Nombre de clients uniques
0,2021-03-01,5676
1,2021-04-01,5674
2,2021-05-01,5644
3,2021-06-01,5659
4,2021-07-01,5672
5,2021-08-01,5642
6,2021-09-01,5693
7,2021-10-01,6190
8,2021-11-01,5875
9,2021-12-01,5867


In [179]:
fig331 = px.line(df_nb_customers, x="date_mois", y="Nombre de clients uniques", title="Evolution du nombre de clients uniques par mois")
fig331.update_layout(
    xaxis_title="Temps"
)
fig331.show()

##3.4 - Nombre de transactions

In [180]:
#Groupby par date et nombre de sessions aboutissant à un achat
df_dates_nb_transactions=df_transactions_produits.groupby(['date_mois'])['session_id'].nunique().reset_index()
df_dates_nb_transactions.rename(columns={'session_id' : 'Nombre de transactions'},inplace=True)

display(df_dates_nb_transactions)

,date_mois,Nombre de transactions
0,2021-03-01,14201
1,2021-04-01,13970
2,2021-05-01,14168
3,2021-06-01,13838
4,2021-07-01,13571
5,2021-08-01,13591
6,2021-09-01,14848
7,2021-10-01,14781
8,2021-11-01,14605
9,2021-12-01,15566


In [181]:
fig341 = px.line(df_dates_nb_transactions, x="date_mois", y="Nombre de transactions", title="Evolution du nombre de transactions dans le temps")
fig341.update_layout(
    xaxis_title="Temps"
)
fig341.show()

##3.5 - Nombre de produits vendus

In [182]:
print("Le Nombre d'articles vendus est de {}".format(len(df_transactions_produits)))

Le Nombre d'articles vendus est de 687534


In [183]:
#Groupby par dates et nombre de produits vendus
df_dates_quantites_vendues=df_transactions_produits.groupby('date_mois')['id_prod'].count().reset_index()
df_dates_quantites_vendues.rename(columns={'id_prod' : "Nombre d'articles vendus"},inplace=True)

In [184]:
fig351 = px.line(df_dates_quantites_vendues, x="date_mois", y="Nombre d'articles vendus", title="Evolution du nombre d'articles vendus dans le temps")
fig351.update_layout(
    xaxis_title="Temps"
)
fig351.show()

##3.6.1 - Tops

In [185]:
#Groupby par produit et CA
df_quantites_vendues_produits = df_transactions_produits.groupby('id_prod')['price'].sum().reset_index()
df_quantites_vendues_produits.rename(columns={'price' : 'CA'},inplace=True)

In [186]:
df_quantites_vendues_produits.sort_values(['CA'],ascending=False, inplace=True)

fig361 = px.bar(df_quantites_vendues_produits.head(10), x='id_prod', y='CA', title='Top 10 Articles par CA')
fig361.update_layout(
    xaxis_title="Id du Produit",
    yaxis_title="CA(€)",
)
fig361.show()

##3.6.2 - Flops

In [187]:
df_quantites_vendues_produits.sort_values(['CA'],ascending=True, inplace=True)

fig362 = px.bar(df_quantites_vendues_produits.head(10), x='id_prod', y='CA', title='Flop 10 Articles par CA')
fig362.update_layout(
    xaxis_title="Id du Produit",
    yaxis_title="CA(€)",
)
fig362.show()

##3.7 - Répartition par catégorie

In [188]:
#Comptons le nombre de produits au sein de chaque catégorie
df_repartition_categ=df_products.groupby(['categ'])['id_prod'].count().reset_index()
df_repartition_categ.rename(columns={'id_prod' : 'Nombre de produits'},inplace=True)

In [189]:
fig371 = px.bar(df_repartition_categ, x="categ", y="Nombre de produits", title="Répartition des produits par catégorie")
fig371.show()

In [190]:
fig372 = px.pie(df_repartition_categ, values='Nombre de produits', names='categ', title="Répartition des produits par catégorie")
fig372.show()

##3.8 - Analyses des clients

###3.8.1 - Top meilleurs clients

In [191]:
##Groupby permettant de mettre en avant les dépenses annuelles des clients
df_top_clients = df_transactions_produits.groupby('client_id')['price'].sum().reset_index()
df_top_clients.rename(columns={'price' : 'CA'},inplace=True)
df_top_clients=df_top_clients.sort_values(by = 'CA', ascending = False)
display(df_top_clients)

,client_id,CA
677,c_1609,326039.89
4388,c_4958,290227.03
6337,c_6714,153918.60
2724,c_3454,114110.57
634,c_1570,5285.82
...,...,...
3855,c_4478,13.36
4044,c_4648,11.20
7889,c_8114,9.98
7918,c_8140,8.30


In [192]:
## Graphique barre TOP 10 clients
fig381 = px.bar(df_top_clients.head(10), x="client_id", y="CA", title="Top 10 meilleurs clients")
fig381.update_layout(
    xaxis_title="Id du Client",
    yaxis_title="CA(€)",
)
fig381.show()

##Les dépenses des 4 premiers sont bien trop importantes pour des clients individuels
## c_1609, c_4958,c_6714, c_3454 sont des clients BToB

###3.8.2 - Répartition du CA pour les clients BToB

In [193]:
##Filtrage des clients BToB
df_BToB = df_top_clients.loc[df_top_clients['client_id'].isin(['c_1609', 'c_4958','c_6714', 'c_3454'])]

##Ajout de la part de leurs dépenses dans le CA
df_BToB['part_CA(%)']=round(df_BToB['CA']/CA_total,2)*100

display(df_BToB)

##Nous écarterons ces clients pour l'étude des corrélations

/tmp/ipython-input-3023663311.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,client_id,CA,part_CA(%)
677,c_1609,326039.89,3.0
4388,c_4958,290227.03,2.0
6337,c_6714,153918.60,1.0
2724,c_3454,114110.57,1.0


###3.8.3 - Courbe de Lorenz

In [194]:
#Tri ascendant des dépenses clients
df_Lorenz_clients=df_top_clients.sort_values(by = 'CA', ascending = True)

##Calcul des indicateurs cumulés
df_Lorenz_clients['depenses_cum']=df_Lorenz_clients['CA'].cumsum()
df_Lorenz_clients['depenses_cum_percentage']=df_Lorenz_clients['depenses_cum']/CA_total
df_Lorenz_clients['population_cum']=(np.arange(1, len(df_Lorenz_clients) + 1)) / len(df_Lorenz_clients)

display(df_Lorenz_clients)

,client_id,CA,depenses_cum,depenses_cum_percentage,population_cum
8151,c_8351,6.31,6.31,5.246239e-07,0.000116
7918,c_8140,8.30,14.61,1.214700e-06,0.000233
7889,c_8114,9.98,24.59,2.044454e-06,0.000349
4044,c_4648,11.20,35.79,2.975640e-06,0.000465
3855,c_4478,13.36,49.15,4.086413e-06,0.000581
...,...,...,...,...,...
634,c_1570,5285.82,11143367.01,9.264781e-01,0.999535
2724,c_3454,114110.57,11257477.58,9.359655e-01,0.999651
6337,c_6714,153918.60,11411396.18,9.487625e-01,0.999767
4388,c_4958,290227.03,11701623.21,9.728925e-01,0.999884


In [195]:
## Tracer la courbe
fig383= px.line(df_Lorenz_clients, x="population_cum", y="depenses_cum_percentage", title="Courbe de Lorenz", width=800, height=800)
fig383.update_layout(
    xaxis_title="Pourcentage cumulé de la population client",
    yaxis_title="Pourcentage cumulé du CA",
)
# Add the line of perfect equality (y=x)
fig383.add_scatter(x=[0, 1], y=[0, 1], mode='lines', name='Perfect Equality', line=dict(color='green', dash='dash'))
fig383.show()

#4 - Corrélations

In [196]:
#Nouveau dataframe contenant toutes les données, sans les données BToB
df_global=pd.merge(df_transactions_produits, df_customers, on='client_id', how='left')
df_global.drop( df_global[ df_global['client_id'].isin(['c_1609', 'c_4958','c_6714', 'c_3454']) ].index, inplace=True)
df_global.info()

<class 'pandas.core.frame.DataFrame'>
Index: 640734 entries, 0 to 687533
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     640734 non-null  object        
 1   date        640734 non-null  datetime64[ns]
 2   session_id  640734 non-null  object        
 3   client_id   640734 non-null  object        
 4   date_mois   640734 non-null  datetime64[ns]
 5   date_annee  640734 non-null  datetime64[ns]
 6   date_jour   640734 non-null  datetime64[ns]
 7   price       640734 non-null  float64       
 8   categ       640734 non-null  int64         
 9   sex         640734 non-null  object        
 10  birth       640734 non-null  int64         
dtypes: datetime64[ns](4), float64(1), int64(2), object(4)
memory usage: 58.7+ MB


##4.1 - Genre d’un client et catégories des livres achetés

In [197]:
#Groupby
df_genre_categ=df_global.groupby(['sex','categ'])['id_prod'].count().reset_index()
df_genre_categ.rename(columns={'id_prod' : 'Nombre de produits'},inplace=True)

In [198]:
fig411 = px.histogram(df_genre_categ, x='categ', y='Nombre de produits', color='sex', barmode='group', title="Nombre de produits vendus par catégorie en fonction du sexe")
fig411.update_layout(
    xaxis_title="Catégorie produit",
    yaxis_title="Nombre d'articles vendus",
)
fig411.show()

In [199]:
# Créer le tableau de contingence
contingency_table2 = pd.crosstab(df_global['sex'], df_global['categ'])

# Calculer le test du Chi-2
chi2_stat, p_value, dof, expected = stats.chi2_contingency(contingency_table2)

print(f"Statistique Chi-2: {chi2_stat}")
print(f"Valeur p: {p_value}")
print(f"Degrés de liberté: {dof}")
print("Fréquences attendues:")

print(expected)

Statistique Chi-2: 22.66856665178056
Valeur p: 1.1955928116587024e-05
Degrés de liberté: 2
Fréquences attendues:
[[201574.89662481 114822.13191434  17096.97146086]
 [185706.10337519 105782.86808566  15751.02853914]]


p<0,05 : on rejette l'hypothèse nulle, il y a bien une corrélation entre le genre et la catégorie de produits achetés

##4.2 - Age des clients et montant total des achats

In [200]:
##Calcul de l'âge au moment de l'achat
df_global['age'] = (df_global['date_annee'].dt.year.astype(int) - df_global['birth'])

display(df_global)

,id_prod,date,session_id,client_id,date_mois,date_annee,date_jour,price,categ,sex,birth,age
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,2021-03-01,2021-01-01,2021-03-01,11.99,0,f,1967,54
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,2021-03-01,2021-01-01,2021-03-01,19.37,0,m,1960,61
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,2021-03-01,2021-01-01,2021-03-01,4.50,0,m,1988,33
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,2021-03-01,2021-01-01,2021-03-01,6.55,0,f,1989,32
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,2021-03-01,2021-01-01,2021-03-01,16.49,0,f,1956,65
...,...,...,...,...,...,...,...,...,...,...,...,...
687529,1_508,2023-02-28 23:49:03.148402,s_348444,c_3573,2023-02-01,2023-01-01,2023-02-28,21.92,1,f,1996,27
687530,2_37,2023-02-28 23:51:29.318531,s_348445,c_50,2023-02-01,2023-01-01,2023-02-28,48.99,2,f,1994,29
687531,1_695,2023-02-28 23:53:18.929676,s_348446,c_488,2023-02-01,2023-01-01,2023-02-28,26.99,1,f,1985,38
687532,0_1547,2023-02-28 23:58:00.107815,s_348447,c_4848,2023-02-01,2023-01-01,2023-02-28,8.99,0,m,1953,70


In [201]:
## Groupby -> Montant total des achats par âge
df_age_montant=df_global.groupby('age')['price'].sum().reset_index()
df_age_montant.rename(columns={'price' : 'CA'},inplace=True)

fig422 = px.scatter(df_age_montant, x='age', y='CA', title="Evolution du total d'achats en fonction de l'âge")
fig422.update_layout(
    xaxis_title="Age du client",
    yaxis_title="Total des achats(€)",
)
fig422.show()

In [202]:
# Extraire la colonne 'du total de dépenses
total_achats = df_age_montant['CA'].dropna()


# Effectuer le test de Shapiro-Wilk
stat, p_value = stats.shapiro(total_achats)


# Afficher les résultats
print(f"Statistique du test de Shapiro-Wilk : {stat}")
print(f"Valeur p : {p_value}")


# Interprétation des résultats
alpha = 0.05
if p_value > alpha:
    print("Les données suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données ne suivent pas une distribution normale (on rejette H0)")

Statistique du test de Shapiro-Wilk : 0.932272072519602
Valeur p : 0.00045547685344637766
Les données ne suivent pas une distribution normale (on rejette H0)


In [203]:
# Extraire les colonnes pertinentes
age = df_age_montant['age']
total_achats = df_age_montant['CA']

# Calculer le coefficient de corrélation de Spearman et la valeur p
spearman_corr, spearman_p_value = stats.spearmanr(age, total_achats)

print(f"Coefficient de corrélation de Spearman: {spearman_corr}")
print(f"Valeur p: {spearman_p_value}")

Coefficient de corrélation de Spearman: -0.88042337409426
Valeur p: 2.4326850391487356e-26


p<0,05, on rejette l'hypothèse H0 : il y a bien une corrélation entre l'âge et le montant des achats

Le coefficient indique une corrélation forte corrélation négative

##4.3 - Age des clients et fréquence d’achat

In [204]:
#Calcul du nombre de sessions par mois en fonction de l'âge
df_age_nombre_transactions=df_global.groupby(['age','date_mois'])['session_id'].nunique().reset_index()
df_age_nombre_transactions.rename(columns={'session_id' : 'Nombre de transactions'},inplace=True)

#Calcul de la fréquence mensuelle de sessions moyenne en fonction de l'âge
df_age_frequence_moyenne=df_age_nombre_transactions.groupby(['age'])['Nombre de transactions'].mean().reset_index()
df_age_frequence_moyenne.rename(columns={"Nombre de transactions" : "Fréquence mensuelle d'achats"},inplace=True)


#Plot
fig431 = px.scatter(df_age_frequence_moyenne, x='age', y="Fréquence mensuelle d'achats", title="Evolution de la fréquence mensuelle d'achat en fonction de l'âge")
fig431.update_layout(
    xaxis_title="Age du client",
)
fig431.show()

In [205]:
# Extraire la colonne du nombre d'articles moyen par panier
frequencemoyenne = df_age_frequence_moyenne["Fréquence mensuelle d'achats"].dropna()


# Effectuer le test de Shapiro-Wilk
stat, p_value = stats.shapiro(frequencemoyenne)


# Afficher les résultats
print(f"Statistique du test de Shapiro-Wilk : {stat}")
print(f"Valeur p : {p_value}")


# Interprétation des résultats
alpha = 0.05
if p_value > alpha:
    print("Les données suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données ne suivent pas une distribution normale (on rejette H0)")

Statistique du test de Shapiro-Wilk : 0.9335964066484949
Valeur p : 0.0005311813097651152
Les données ne suivent pas une distribution normale (on rejette H0)


In [206]:
# Extraire les colonnes pertinentes
age = df_age_frequence_moyenne['age']
frequencemoyenne = df_age_frequence_moyenne["Fréquence mensuelle d'achats"]

# Calculer le coefficient de corrélation de Spearman et la valeur p
spearman_corr, spearman_p_value = stats.spearmanr(age, frequencemoyenne)

print(f"Coefficient de corrélation de Spearman: {spearman_corr}")
print(f"Valeur p: {spearman_p_value}")

Coefficient de corrélation de Spearman: -0.6780181843472982
Valeur p: 9.170807430574563e-12


p<0,05, on rejette l'hypothèse H0 : il y a bien une corrélation entre l'âge et le montant des achats

Le coefficient indique une corrélation négative

##4.41 - Age des clients et taille du panier moyen

In [207]:
#Calcul du nombre d'articles par panier en fonction de l'âge
df_age_nbarticlespanier=df_global.groupby(['age','session_id'])['id_prod'].count().reset_index()
df_age_nbarticlespanier.rename(columns={'id_prod' : "Nombre d'articles par panier"},inplace=True)

#Calcul du nombre d'articles moyen par panier en fonction de l'âge
df_age_paniermoyen=df_age_nbarticlespanier.groupby(['age'])["Nombre d'articles par panier"].mean().reset_index()
df_age_paniermoyen.rename(columns={"Nombre d'articles par panier" : "Nombre d'articles moyen par panier"},inplace=True)


#Plot
fig441 = px.scatter(df_age_paniermoyen, x='age', y="Nombre d'articles moyen par panier", title="Evolution de la taille du panier moyen en fonction de l'âge")
fig441.update_layout(
    xaxis_title="Age du client",
)
fig441.show()

In [208]:
# Extraire la colonne du nombre d'articles moyen par panier
paniermoyen = df_age_paniermoyen["Nombre d'articles moyen par panier"].dropna()


# Effectuer le test de Shapiro-Wilk
stat, p_value = stats.shapiro(paniermoyen)


# Afficher les résultats
print(f"Statistique du test de Shapiro-Wilk : {stat}")
print(f"Valeur p : {p_value}")


# Interprétation des résultats
alpha = 0.05
if p_value > alpha:
    print("Les données suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données ne suivent pas une distribution normale (on rejette H0)")

Statistique du test de Shapiro-Wilk : 0.7832225263342287
Valeur p : 2.2850668312439762e-09
Les données ne suivent pas une distribution normale (on rejette H0)


In [209]:
# Extraire les colonnes pertinentes
age = df_age_paniermoyen['age']
taille_panier_moyen = df_age_paniermoyen["Nombre d'articles moyen par panier"]

# Calculer le coefficient de corrélation de Spearman et la valeur p
spearman_corr, spearman_p_value = stats.spearmanr(age, taille_panier_moyen)

print(f"Coefficient de corrélation de Spearman: {spearman_corr}")
print(f"Valeur p: {spearman_p_value}")

Coefficient de corrélation de Spearman: -0.5751247178551578
Valeur p: 3.644316308793808e-08


p<0,05 : on rejette H0, il y a bien une corrélation entre l'âge et la taille du panier moyen

Le coefficient indique une forte corrélation négative

##4.42 - Age des clients et valeur du panier moyen

In [210]:
#Calcul du nombre d'articles par panier en fonction de l'âge
df_age_valeurpanier=df_global.groupby(['age','session_id'])['price'].sum().reset_index()
df_age_valeurpanier.rename(columns={'price' : "Valeur du panier"},inplace=True)

#Calcul du nombre d'articles moyen par panier en fonction de l'âge
df_age_valeurpaniermoyen=df_age_valeurpanier.groupby(['age'])["Valeur du panier"].mean().reset_index()
df_age_valeurpaniermoyen.rename(columns={"Valeur du panier" : "Valeur moyenne du panier"},inplace=True)


#Plot
fig442 = px.scatter(df_age_valeurpaniermoyen, x='age', y="Valeur moyenne du panier", title="Evolution de la valeur du panier moyen en fonction de l'âge")
fig442.update_layout(
    xaxis_title="Age du client",
)
fig442.show()

In [211]:
# Extraire la colonne du nombre d'articles moyen par panier
valeurpaniermoyen = df_age_valeurpaniermoyen["Valeur moyenne du panier"].dropna()


# Effectuer le test de Shapiro-Wilk
stat, p_value = stats.shapiro(valeurpaniermoyen)


# Afficher les résultats
print(f"Statistique du test de Shapiro-Wilk : {stat}")
print(f"Valeur p : {p_value}")


# Interprétation des résultats
alpha = 0.05
if p_value > alpha:
    print("Les données suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données ne suivent pas une distribution normale (on rejette H0)")

Statistique du test de Shapiro-Wilk : 0.640316328846843
Valeur p : 1.5846450037678653e-12
Les données ne suivent pas une distribution normale (on rejette H0)


In [212]:
# Extraire les colonnes pertinentes
age = df_age_valeurpaniermoyen['age']
valeur_panier_moyen = df_age_valeurpaniermoyen["Valeur moyenne du panier"]

# Calculer le coefficient de corrélation de Spearman et la valeur p
spearman_corr, spearman_p_value = stats.spearmanr(age, valeur_panier_moyen)

print(f"Coefficient de corrélation de Spearman: {spearman_corr}")
print(f"Valeur p: {spearman_p_value}")

Coefficient de corrélation de Spearman: -0.7561425915856296
Valeur p: 1.2074283210747014e-15


p<0,05 : on rejette H0, il y a bien une corrélation entre l'âge et la valeur du panier moyen

Le coefficient indique une forte corrélation négative

##4.5 - Age des clients et catégorie des livres achetés

In [213]:
#Visualisation des deux paramètres
fig451 = px.box(df_global, x="categ", y="age")
fig451.update_layout(
    yaxis_title="Age du client",
    xaxis_title="Catégorie du produit"
)
fig451.show()

In [214]:
# Testons la normalité sur le groupe Cat0
cat0 = df_global[df_global['categ'] == 0]['age'].dropna()

# Effectuer le test de Shapiro-Wilk
stat, p_value = stats.shapiro(cat0)

# Afficher les résultats
print(f"Statistique du test de Shapiro-Wilk pour Catégorie 0 : {stat}")
print(f"Valeur p pour Catégorie 0 : {p_value}")

# Interprétation des résultats
alpha = 0.05
if p_value > alpha:
    print("Les données de Catégorie 0 suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données de Catégorie 0 ne suivent pas une distribution normale (on rejette H0)")


# Idem pour la catégorie 1
cat1 = df_global[df_global['categ'] == 1]['age'].dropna()
stat, p_value = stats.shapiro(cat1)
print(f"\nStatistique du test de Shapiro-Wilk pour Catégorie 1 : {stat}")
print(f"Valeur p pour Catégorie 1 : {p_value}")
if p_value > alpha:
    print("Les données de Catégorie 1 suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données de Catégorie 1 ne suivent pas une distribution normale (on ne rejette pas H0)")

# Idem pour la catégorie 2
cat2 = df_global[df_global['categ'] == 2]['age'].dropna()
stat, p_value = stats.shapiro(cat2)
print(f"\nStatistique du test de Shapiro-Wilk pour Catégorie 2 : {stat}")
print(f"Valeur p pour Catégorie 2 : {p_value}")
if p_value > alpha:
    print("Les données de Catégorie 2 suivent une distribution normale (on ne rejette pas H0)")
else:
    print("Les données de Catégorie 2 ne suivent pas une distribution normale (on ne rejette pas H0)")


Statistique du test de Shapiro-Wilk pour Catégorie 0 : 0.9355712170760366
Valeur p pour Catégorie 0 : 2.6603716146399467e-126
Les données de Catégorie 0 ne suivent pas une distribution normale (on rejette H0)

Statistique du test de Shapiro-Wilk pour Catégorie 1 : 0.9888442243451404
Valeur p pour Catégorie 1 : 3.436024768283932e-72
Les données de Catégorie 1 ne suivent pas une distribution normale (on ne rejette pas H0)

Statistique du test de Shapiro-Wilk pour Catégorie 2 : 0.6827797953874931
Valeur p pour Catégorie 2 : 3.378919790381429e-117
Les données de Catégorie 2 ne suivent pas une distribution normale (on ne rejette pas H0)


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning:

scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 387281.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning:

scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 220605.

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning:

scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 32848.



In [215]:
#Pas de ditribution normale, il faut utiliser Kruskal_Wallis

#On utilise les 3 colonnes extraites précédemment pour les tests

# Calculer le test Kruskal et la valeur p
H_stat, p_value = stats.kruskal(cat0, cat1, cat2)

print(f"Statistique de test H: {H_stat}")
print(f"Valeur p: {p_value}")

Statistique de test H: 71283.57583245283
Valeur p: 0.0


p<0,05 : on rejette H0, il y a bien une corrélation entre l'âge et la catégorie des produits achetés

Le coefficient indique une très forte corrélation